# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mn1tchA/MLOps/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import duckdb
from google.colab import userdata

# Safely load Hugging Face token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# Define path for our specific mid-panel time window (March 2026)
path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/**/*.parquet"

## 1. Unit of analysis + time window

**Unit of Analysis (Grain):** One row = one report date, per pseudonymized client, per pseudonymized content item.
**Time Window:** `month=2026-03` (March 1 to March 31, 2026).
**Output:** The analysis will hand the human editor a ranked queue of content items most likely to experience a traffic decline, allowing them to prioritize reviews efficiently.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*   **Context:** `report_date`, `client_id`, `content_id`. Used strictly for grouping and joining; never for the model to learn from.
*   **Label / proxy:** A binary flag (future 30-day traffic drop). This is the proxy we predict; it is never a feature.
*   **Excluded:** `is_declining_label`. *Why:* This is a pre-computed product-decision flag derived from future information. Including it triggers the leakage trap (the model memorizes the formula instead of learning the signals).
*   **Features (The Five Honest Features):**
    1. `gsc_impressions`: Knowable at the decision moment because past visibility is already logged.
    2. `gsc_clicks`: Knowable at the decision moment because it relies purely on historical actions prior to the prediction date.
    3. `ctr` (Click-Through Rate): Knowable at the decision moment as a simple computed ratio (`gsc_clicks / gsc_impressions`).
    4. `gsc_avg_position`: Knowable at the decision moment because daily search ranks are recorded at the end of each day.
    5. `dayofweek`: Knowable at the decision moment because the calendar is a static fact.

In [6]:
# Build a small feature frame using only our honest feature bucket
print("--- Honest Feature Frame (First 5 Rows) ---")
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions AS feature_1,
    gsc_clicks AS feature_2,
    (gsc_clicks * 1.0 / gsc_impressions) AS feature_3_ctr, -- Calculated on the fly!
    gsc_avg_position AS feature_4,
    dayofweek(report_date) AS feature_5
FROM read_parquet('{path}')
WHERE gsc_impressions > 0
LIMIT 5
""").show()

--- Honest Feature Frame (First 5 Rows) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬───────────┬───────────┬───────────────┬───────────────────┬───────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ feature_1 │ feature_2 │ feature_3_ctr │     feature_4     │ feature_5 │
│    date     │         varchar         │         varchar          │   int64   │   int64   │    double     │      double       │   int64   │
├─────────────┼─────────────────────────┼──────────────────────────┼───────────┼───────────┼───────────────┼───────────────────┼───────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │        20 │         0 │           0.0 │              3.35 │         0 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_05597932fe4da067 │         1 │         0 │           0.0 │               0.0 │         0 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_7a105f548d9c6916 │       125 │         1 │         0.008 │             4.928 │         0 │
│ 2026-03-01 

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# 1. GRAIN CHECK (Using the exact formula from the skill file)
print("--- Query 1: Grain Check (Should return 0 rows) ---")
con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
FROM read_parquet('{path}')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING c > 1
LIMIT 5
""").show()

# 2. WINDOWS & COUNTS
print("--- Query 2: Date Span and Row Count ---")
con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM read_parquet('{path}')
""").show()

# 3. MISSINGNESS / AVAILABILITY (Filtering with IS TRUE)
print("--- Query 3: Availability (Rows with actual impressions) ---")
con.sql(f"""
SELECT COUNT(*) AS rows_with_impressions
FROM read_parquet('{path}')
WHERE (gsc_impressions > 0) IS TRUE
""").show()

# 4. THE DELIBERATE LEAK EXPERIMENT (The Trap)
print("\n--- The Trap: Feature Leakage ---")
try:
    con.sql(f"""
    SELECT
        gsc_clicks,
        is_declining_label -- THE TRAP: Including the target in the features
    FROM read_parquet('{path}')
    LIMIT 1
    """).show()
except duckdb.BinderException:
    print("Trap avoided: The 'is_declining_label' was deliberately excluded from our working feature frame to prevent the model from artificially achieving a perfect score by looking at the future.")

--- Query 1: Grain Check (Should return 0 rows) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

--- Query 2: Date Span and Row Count ---
┌────────────┬────────────┬────────────┐
│ start_date │  end_date  │ total_rows │
│    date    │    date    │   int64    │
├────────────┼────────────┼────────────┤
│ 2026-03-01 │ 2026-03-31 │    9841378 │
└────────────┴────────────┴────────────┘

--- Query 3: Availability (Rows with actual impressions) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────────┐
│ rows_with_impressions │
│         int64         │
├───────────────────────┤
│               3611061 │
└───────────────────────┘


--- The Trap: Feature Leakage ---
Trap avoided: The 'is_declining_label' was deliberately excluded from our working feature frame to prevent the model from artificially achieving a perfect score by looking at the future.


## 4. Data limits

**Unbalanced History:** As defined in the data warehouse documentation, the history depth per client differs significantly (`dim_clients.gsc_data_start`). This means our dataset has a complete history only for older clients. For newer clients, earlier history is absent, which will make their trailing historical averages highly volatile and harder for the model to confidently score.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.